# Phase 3 - Notebook 04: DUSt3R Complete Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/04_dust3r_architecture.ipynb)


## Setup


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("Setup complete!")

## 1. DUSt3R 完整架构

```
Image1 (H×W×3)          Image2 (H×W×3)
      |                       |
      v                       v
┌─────────────┐         ┌─────────────┐
│ ViT Encoder │         │ ViT Encoder │
│(共享权重)    │         │(共享权重)    │
└─────────────┘         └─────────────┘
      |                       |
      v                       v
 Features1 (N×D)    Features2 (N×D)
      \                      /
       → Cross-Attention Decoder
         ├── 自注意力
         ├── 交叉注意力
         └── 融合特征
              |        |
              v        v
            Head1    Head2
              |        |
              v        v
         Pointmap1  Pointmap2
        Confidence1 Confidence2
```


In [ ]:
# 绘制 DUSt3R 架构图
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('DUSt3R Complete Architecture', fontsize=16, fontweight='bold')

# 输入图像
colors = {
    'input': '#E8EAF6',
    'encoder': '#E3F2FD',
    'decoder': '#F3E5F5',
    'head': '#E8F5E9',
    'output': '#FFF9C4'
}

# 左侧：Image 1 路径
boxes_left = [
    (0.5, 9.5, 3, 1.5, 'Image 1\n(H×W×3)', colors['input']),
    (0.5, 7, 3, 1.5, 'ViT Encoder\n(Shared)', colors['encoder']),
    (0.5, 4, 3, 1.5, 'Features 1\n(N×D)', colors['encoder']),
]

for x, y, w, h, text, color in boxes_left:
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, fontweight='bold')

# 右侧：Image 2 路径
boxes_right = [
    (10.5, 9.5, 3, 1.5, 'Image 2\n(H×W×3)', colors['input']),
    (10.5, 7, 3, 1.5, 'ViT Encoder\n(Shared)', colors['encoder']),
    (10.5, 4, 3, 1.5, 'Features 2\n(N×D)', colors['encoder']),
]

for x, y, w, h, text, color in boxes_right:
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, fontweight='bold')

# 中央：Decoder
decoder_box = FancyBboxPatch((4, 2.5), 6, 2, boxstyle='round,pad=0.15',
                             facecolor=colors['decoder'], edgecolor='#6A1B9A', linewidth=2)
ax.add_patch(decoder_box)
ax.text(7, 3.5, 'Cross-Attention Decoder', ha='center', va='center',
        fontsize=12, fontweight='bold')
ax.text(7, 2.9, '• Self-Attention\n• Cross-Attention\n• Feature Fusion',
        ha='center', va='center', fontsize=8)

# 箭头：输入到编码器
for x, y in [(2, 9.5), (12, 9.5)]:
    ax.annotate('', xy=(x, 8.5), xytext=(x, 9.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# 箭头：编码器到特征
for x, y in [(2, 7), (12, 7)]:
    ax.annotate('', xy=(x, 5.5), xytext=(x, 7),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# 箭头：特征到解码器
ax.annotate('', xy=(5.5, 3.5), xytext=(3.5, 4.5),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=2))
ax.annotate('', xy=(8.5, 3.5), xytext=(10.5, 4.5),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=2))

# 输出头
heads = [
    (1, 0.5, 3, 1.2, 'Head 1', colors['head']),
    (5, 0.5, 3, 1.2, 'Head 1
(异形)', colors['head']),
    (9, 0.5, 3, 1.2, 'Head 2', colors['head']),
]

for x, y, w, h, text, color in heads:
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9, fontweight='bold')

# 箭头：解码器到头
ax.annotate('', xy=(2.5, 1.7), xytext=(5, 2.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(6.5, 1.7), xytext=(7, 2.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(10.5, 1.7), xytext=(9, 2.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

plt.tight_layout()
plt.savefig('dust3r_architecture.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. 非对称解码器

DUSt3R 使用两个独立的解码器：
- Head 1：预测 Image 1 中每个像素的 3D 坐标
- Head 2：预测 Image 2 中每个像素的 3D 坐标（在同一坐标系）


In [ ]:
class CrossAttentionDecoder(nn.Module):
    """简化的交叉注意力解码器。"""
    
    def __init__(self, embed_dim, num_heads=8, num_layers=4):
        super().__init__()
        self.embed_dim = embed_dim
        
        # 多个解码器块
        self.decoder_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=embed_dim * 4,
                batch_first=True,
                activation='gelu'
            )
            for _ in range(num_layers)
        ])
        
        # 输出头：预测 3D 坐标 + 置信度
        self.head_3d = nn.Linear(embed_dim, 3)  # 3D 坐标
        self.head_conf = nn.Linear(embed_dim, 1)  # 置信度
    
    def forward(self, feat1, feat2):
        # feat1, feat2: [B, N, D]
        # 简化：以 feat1 为主，用 feat2 进行交叉注意力
        
        # 这里简化处理——实际 DUSt3R 更复杂
        x = feat1 + feat2  # 简单融合
        
        # 通过解码器块
        for block in self.decoder_blocks:
            x = block(x)
        
        # 预测输出
        pointmap = self.head_3d(x)  # [B, N, 3]
        confidence = self.head_conf(x).sigmoid()  # [B, N, 1]
        
        return pointmap, confidence

print("CrossAttentionDecoder 定义完成")

## 3. CroCo 预训练基础

DUSt3R 基于 CroCo (Cross-View Completion) 预训练：

```
CroCo 预训练：
  输入：两张图像
  任务：用 Image 1 的信息重建 Image 2 的 3D 几何
  目标：学习跨视角的几何对应关系
```

这种自监督预训练使网络学习到有用的 3D 几何表示。


In [ ]:
# 演示 DUSt3R 完整推理流程
print("DUSt3R 推理流程（概念演示）：")
print()
print("Step 1: ViT 编码器")
print(f"  Input: Image1 [B, 3, 512, 512], Image2 [B, 3, 512, 512]")
print(f"  Output: Features1 [B, 1024, 1024], Features2 [B, 1024, 1024]")
print()
print("Step 2: 交叉注意力解码器")
print(f"  Input: Features1, Features2")
print(f"  Process: 4 个解码器块 (自注意力 + 交叉注意力)")
print()
print("Step 3: 预测头（非对称）")
print(f"  Head1 → Pointmap1 [B, H, W, 3] + Confidence1 [B, H, W]")
print(f"  Head2 → Pointmap2 [B, H, W, 3] + Confidence2 [B, H, W]")
print()
print("Step 4: 后处理")
print(f"  使用 Procrustes 分析估计相对位姿")
print(f"  可选：全局位姿优化")

## 4. Loss 函数设计

DUSt3R 使用组合损失：

1. **回归损失**：Pointmap 与 GT 的 L1 差异
2. **置信度损失**：鼓励网络对不确定区域预测低置信度


In [ ]:
def dust3r_loss(pointmap_pred, pointmap_gt, confidence_pred, confidence_gt):
    """
    DUSt3R 损失函数（简化版本）
    """
    # 回归损失：加权 L1 损失
    point_diff = torch.abs(pointmap_pred - pointmap_gt).mean(dim=-1)  # [B, H, W]
    regression_loss = (point_diff * confidence_pred.squeeze(-1)).mean()
    
    # 置信度损失：鼓励高置信度在难区域低
    confidence_loss = (-torch.log(confidence_pred + 1e-8) * (point_diff > 1.0)).mean()
    
    # 总损失
    total_loss = regression_loss + 0.1 * confidence_loss
    
    return {
        'total': total_loss,
        'regression': regression_loss,
        'confidence': confidence_loss
    }

print("DUSt3R loss 函数定义完成")

## 5. 与 SLAM 和 MVS 的联系

| 方面 | SLAM | MVS | DUSt3R |
|------|------|-----|--------|
| 输入 | 视频流 | 多张图像 | 图像对 |
| 输出 | 轨迹 + 稀疏/密集重建 | 密集点云 | Pointmaps |
| 相机估计 | 逐帧优化 | 全局 BA | 端到端神经网络 |
| 速度 | 实时 | 分钟级 | <1秒 |


## 6. Summary

**Key Concepts:**
1. ViT 编码器提取图像特征
2. 交叉注意力解码器进行视图间交互
3. 非对称输出头预测两个 Pointmaps
4. CroCo 预训练学习 3D 几何
5. 组合损失优化 Pointmap 和置信度

---

**Next**: [05_mast3r_extension.ipynb](./05_mast3r_extension.ipynb)
